# AmsiNet - Step 10: (a) direct test of the "context pulls words towards Latin" explanation, (b) disagreement gate for the region agents

**Runtime:** Colab **CPU**. Standalone: it reloads the saved probabilities of Step 3a (five seeds, both word heads). Nothing is trained.

**(b) The gate.** Step 9 showed that the line and block agents help on single-script lines (+5.5 points) and hurt on mixed-script lines (about -1.8 points). A region agent is therefore added to a word only if the words of that region *agree* about their script: for a line (block) let `d` be 1 minus the share of its words whose current predicted script equals the region's majority prediction; the agent is used only if `d <= rho`. `rho = 1` means no gate (the Step 9 system). For every seed and word head, the fusion weights are refitted under each `rho` on the validation set, `rho` is chosen by validation macro-F1, `tau*` follows the usual 90% rule, and the test set is evaluated once. The gate also skips the agent calls it blocks, so it saves compute.

In [ ]:
# 1. Setup
from google.colab import drive
import os
if not os.path.ismount('/content/drive'): drive.mount('/content/drive')
assert os.path.exists('/content/drive/MyDrive/amsinet/data/bstd_index.json'), 'Drive is not mounted or amsinet/data is missing: reconnect and run this cell again'
import json, numpy as np, pandas as pd
from scipy.special import softmax
from scipy import stats
ROOT = '/content/drive/MyDrive/amsinet'
CFG = dict(SEEDS=[0, 1, 2, 3, 4], VARIANTS=['word', 'word_moe'], EPS=1e-4, KEEP=0.90)
D = json.load(open(f'{ROOT}/data/bstd_index.json', encoding='utf-8'))
SCRIPTS, INDEX = D['scripts'], D['index']; SIDX = {s: i for i, s in enumerate(SCRIPTS)}; C = len(SCRIPTS); IDX = {r['id']: r for r in INDEX}
META = pd.read_csv(f'{ROOT}/data/crops_meta.csv'); REGDF = pd.read_csv(f'{ROOT}/data/regions_meta.csv')
eff = pd.read_csv(f'{ROOT}/agents/agent_efficiency_cpu.csv').set_index('agent')['GFLOPs']
flat_df = pd.read_csv(f'{ROOT}/results/backbone_efficiency.csv'); FLAT = float(flat_df[flat_df.agent == 'Flat baseline']['GFLOPs'].iloc[0])
missing = [f'{t}_seed{s}' for s in CFG['SEEDS'] for t in ('block', 'line', 'word', 'word_moe') if not os.path.exists(f'{ROOT}/agents/{t}_seed{s}_test_probs.npy')]
print('missing agent files:', missing or 'none')

In [ ]:
# 2. Helpers (same definitions as Steps 6 and 8)
def load(tag, sp):
    P = np.load(f'{ROOT}/agents/{tag}_{sp}_probs.npy').astype(np.float32); pa = pd.read_csv(f'{ROOT}/agents/{tag}_{sp}_paths.csv'); return P, pa.path.tolist()
def region_lookup(level, tag, sp):
    P, paths = load(tag, sp); pos = {p: i for i, p in enumerate(paths)}
    d = REGDF[(REGDF.level == level) & (REGDF.split == sp)]
    return {(r.image_id, r.rid): P[pos[r.path]] for r in d.itertuples() if r.path in pos}
def table(sp, wtag, ltag, btag):
    d = META[META.split == sp].reset_index(drop=True); n = len(d)
    Pw, pw = load(wtag, sp); assert pw == d.path.tolist(), 'word probabilities are not aligned with crops_meta.csv'
    logP = np.log(np.clip(Pw, CFG['EPS'], 1)); lid, bid = [], []
    for iid, k in zip(d.image_id, d.word_idx):
        w = IDX[iid]['words'][k]; lid.append(w.get('line', -1)); bid.append(w.get('block', -1))
    img = d.image_id.values; lkey = img.astype(str) + '|l' + np.array(lid).astype(str); bkey = img.astype(str) + '|b' + np.array(bid).astype(str)
    lg, lu = pd.factorize(lkey); bg, bu = pd.factorize(bkey)
    def loo(g, ng):
        S = np.zeros((ng, C)); cnt = np.zeros(ng); np.add.at(S, g, logP); np.add.at(cnt, g, 1); o = cnt[g] - 1
        F = np.zeros((n, C)); m = o > 0; F[m] = (S[g][m] - logP[m]) / o[m][:, None]; return F
    def reg(level, tag, key):
        L = region_lookup(level, tag, sp); F = np.zeros((n, C))
        for i, (iid, kk) in enumerate(zip(img, key)):
            v = L.get((iid, f'{level[0]}{kk}'))
            if v is not None: F[i] = np.log(np.clip(v, CFG['EPS'], 1))
        return F
    ui, inv = np.unique(img, return_inverse=True); l_img = np.zeros(len(lu), int); l_img[lg] = inv; b_img = np.zeros(len(bu), int); b_img[bg] = inv
    return dict(y=np.array([SIDX[s] for s in d.script]), word=logP, line_ctx=loo(lg, len(lu)), block_ctx=loo(bg, len(bu)),
                line_agent=reg('line', ltag, lid), block_agent=reg('block', btag, bid),
                img=img, inv=inv, n_img=len(ui), lg=lg, bg=bg, nl=len(lu), nb=len(bu), l_img=l_img, b_img=b_img)
def confmat(y, p): return np.bincount(y * C + p, minlength=C * C).reshape(C, C)
def f1_from(cm):
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp; den = 2 * tp + fp + fn; return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
def metr(y, p): cm = confmat(y, p); return dict(acc=100 * float(np.trace(cm) / cm.sum()), f1=100 * float(f1_from(cm).mean()))
def unc(s): p = softmax(s, axis=1); return -(p * np.log(np.clip(p, 1e-12, 1))).sum(1) / np.log(C)
KEYS = ['line_ctx', 'block_ctx', 'line_agent', 'block_agent']; GRID_W = [0, .1, .25, .5, 1, 1.5, 2, 3]
def cascade(T, tau, W):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], u > tau); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], u > tau); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def cost(T, li, bi, GF):
    nw = np.bincount(T['inv'], minlength=T['n_img']).astype(float); nl = np.bincount(T['l_img'][li], minlength=T['n_img']).astype(float); nb = np.bincount(T['b_img'][bi], minlength=T['n_img']).astype(float)
    return float((nw * GF['word'] + nl * GF['line'] + nb * GF['block']).mean())
TAUS = [0, .02, .05, .1, .15, .2, .3, .4, .5, .6, .7, .8, .9, 1.01]

In [ ]:
RHOS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 1.0]
def dis_of(g, ng, pred):
    M = np.zeros((ng, C)); np.add.at(M, (g, pred), 1); return (1 - M.max(1) / M.sum(1))[g]          # per word: disagreement of its region
def cascade_g(T, tau, W, rho):
    s = T['word'] + W['line_ctx'] * T['line_ctx'] + W['block_ctx'] * T['block_ctx']; u = unc(s)
    ok_l = dis_of(T['lg'], T['nl'], s.argmax(1)) <= rho + 1e-9
    li = np.zeros(T['nl'], bool); np.logical_or.at(li, T['lg'], (u > tau) & ok_l); s = s + W['line_agent'] * T['line_agent'] * li[T['lg']][:, None]
    u = unc(s); ok_b = dis_of(T['bg'], T['nb'], s.argmax(1)) <= rho + 1e-9
    bi = np.zeros(T['nb'], bool); np.logical_or.at(bi, T['bg'], (u > tau) & ok_b); s = s + W['block_agent'] * T['block_agent'] * bi[T['bg']][:, None]
    return s, li, bi
def fit_W(TV, rho):
    f = lambda W: metr(TV['y'], cascade_g(TV, 0, W, rho)[0].argmax(1))['f1']; W = {k: 0.0 for k in KEYS}
    for _ in range(2):
        for k in KEYS: W[k] = max(GRID_W, key=lambda g: f({**W, k: g}))
    return W, f(W)
def mixed_mask(T):
    first = np.full(T['nl'], -1); mixed = np.zeros(T['nl'], bool)
    for g, yy in zip(T['lg'], T['y']):
        if first[g] < 0: first[g] = yy
        elif first[g] != yy: mixed[g] = True
    return mixed[T['lg']]
def run_g(seed, variant, keep=False):
    GF = dict(word=float(eff[variant]), line=float(eff['line']), block=float(eff['block']))
    wtag, ltag, btag = f'{variant}_seed{seed}', f'line_seed{seed}', f'block_seed{seed}'
    TV, TT = table('val', wtag, ltag, btag), table('test', wtag, ltag, btag); y = TT['y']; mw = mixed_mask(TT)
    fits = {rho: fit_W(TV, rho) for rho in RHOS}
    best = max(RHOS, key=lambda r: (round(fits[r][1], 6), r))                                  # ties -> larger rho (less gating)
    out = dict(seed=seed, variant=variant, rho_star=best)
    def evaluate(rho, tag):
        W = fits[rho][0]
        cv = {t: (lambda r: (metr(TV['y'], r[0].argmax(1))['f1'], cost(TV, r[1], r[2], GF)))(cascade_g(TV, t, W, rho)) for t in TAUS}
        f_full, f_ctx = cv[0][0], cv[1.01][0]; target = f_ctx + CFG['KEEP'] * (f_full - f_ctx)
        ok = [t for t in TAUS if cv[t][0] >= target]; tau = min(ok, key=lambda t: cv[t][1]) if ok else 0.0
        res = {}
        for name, (s, li, bi) in dict(full=cascade_g(TT, 0, W, rho), cascade=cascade_g(TT, tau, W, rho)).items():
            m = metr(y, s.argmax(1)); out[f'{tag}_{name}_acc'], out[f'{tag}_{name}_f1'], out[f'{tag}_{name}_gflops'] = m['acc'], m['f1'], cost(TT, li, bi, GF)
            out[f'{tag}_{name}_acc_mixed'] = 100 * float((s.argmax(1)[mw] == y[mw]).mean()); out[f'{tag}_{name}_acc_single'] = 100 * float((s.argmax(1)[~mw] == y[~mw]).mean()); res[name] = s.argmax(1)
        out[f'{tag}_tau'] = tau; return res
    base = evaluate(1.0, 'ungated'); gated = evaluate(best, 'gated')
    # oracle upper bound (uses the TRUE mixed-script lines; not a deployable system)
    Wf = fits[1.0][0]; s_all = cascade_g(TT, 0, Wf, 1.0)[0].argmax(1); s_w = TT['word'].argmax(1); orc = np.where(mw, s_w, s_all)
    out['oracle_gate_acc'], out['oracle_gate_f1'] = metr(y, orc)['acc'], metr(y, orc)['f1']
    pk = dict(y=y, word=s_w, full=base['full'], gated=gated['full'], lg=TT['lg'], bg=TT['bg']) if keep else None
    return out, pk

## Run: five seeds, both word heads (about 2 minutes per run)

In [ ]:
rows, PK = [], {}
for variant in CFG['VARIANTS']:
    for seed in CFG['SEEDS']:
        o, pk = run_g(seed, variant, keep=(seed == CFG['SEEDS'][0])); rows.append(o)
        if pk is not None: PK[variant] = pk
        print(variant, seed, 'rho* %.1f' % o['rho_star'], '| ungated cascade %.2f/%.2f (%.1f GF) -> gated %.2f/%.2f (%.1f GF) | full: %.2f/%.2f -> %.2f/%.2f | oracle gate %.2f/%.2f' % (
              o['ungated_cascade_acc'], o['ungated_cascade_f1'], o['ungated_cascade_gflops'], o['gated_cascade_acc'], o['gated_cascade_f1'], o['gated_cascade_gflops'],
              o['ungated_full_acc'], o['ungated_full_f1'], o['gated_full_acc'], o['gated_full_f1'], o['oracle_gate_acc'], o['oracle_gate_f1']))
GS = pd.DataFrame(rows); GS.to_csv(f'{ROOT}/results/gate_per_seed.csv', index=False)

## Mean +/- 95% CI over seeds, paired differences, and comparison with the flat baseline

In [ ]:
def mci(x):
    x = np.asarray(x, float); n = len(x); h = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n) if n > 1 else float('nan'); return x.mean(), h
def fmt(x, d=2): m, h = mci(x); return f'{m:.{d}f} +/- {h:.{d}f}'
SUM = []
for variant in CFG['VARIANTS']:
    d = GS[GS.variant == variant]
    for tag, lab in (('ungated', 'cascade, no gate (Step 9)'), ('gated', 'cascade with disagreement gate')):
        SUM.append((variant, lab, fmt(d[f'{tag}_cascade_acc']), fmt(d[f'{tag}_cascade_f1']), fmt(d[f'{tag}_cascade_gflops'], 1), fmt(d[f'{tag}_cascade_acc_single']), fmt(d[f'{tag}_cascade_acc_mixed'])))
    SUM.append((variant, 'oracle gate (true mixed lines; bound only)', fmt(d.oracle_gate_acc), fmt(d.oracle_gate_f1), '-', '-', '-'))
SD = pd.DataFrame(SUM, columns=['word head', 'system', 'accuracy %', 'macro-F1 %', 'GFLOPs / image', 'acc % single-script lines', 'acc % mixed-script lines']); display(SD); SD.to_csv(f'{ROOT}/results/gate_summary.csv', index=False)
print('flat ViT-B/16 (five seeds): 93.50 +/- 0.35 acc | 88.06 +/- 0.60 macro-F1 | 35.13 GFLOPs per word (840.8 per image)')
for variant in CFG['VARIANTS']:
    d = GS[GS.variant == variant]; print('\n', variant, 'rho* per seed:', d.rho_star.tolist(), '| tau* gated:', d.gated_tau.tolist())
    for k in ('cascade_acc', 'cascade_f1', 'cascade_gflops', 'full_acc', 'full_f1', 'cascade_acc_mixed', 'cascade_acc_single'):
        dd = (d[f'gated_{k}'] - d[f'ungated_{k}']).values; m, h = mci(dd); print(f'  gated minus ungated, {k}: {m:+.2f} +/- {h:.2f}')
    for k, fl in (('acc', 93.50), ('f1', 88.06)):
        m, h = mci(d[f'gated_cascade_{k}']); print(f'  gated cascade {k} minus flat mean: {m - fl:+.2f}  (cascade 95% CI half-width {h:.2f}; flat 0.35 / 0.60)')

## (a) Direct test of the pull towards Latin (first seed)

In [ ]:
LAT = next(i for i, s in enumerate(SCRIPTS) if s.lower() in ('latin', 'english', 'roman'))
for VAR in CFG['VARIANTS']:
    p = PK[VAR]; y = p['y']; nonlat = y != LAT
    def nb_share(g, pred):      # leave-one-out share of the other words in the region that the WORD agent calls Latin
        n_g = np.bincount(g); lat_g = np.bincount(g, weights=(pred == LAT).astype(float)); self_l = (pred == LAT).astype(float); o = n_g[g] - 1
        return np.where(o > 0, (lat_g[g] - self_l) / np.maximum(o, 1), np.nan)
    sh_l, sh_b = nb_share(p['lg'], p['word']), nb_share(p['bg'], p['word'])
    G = {'sent to Latin by fusion only (newly wrong)': nonlat & (p['full'] == LAT) & (p['word'] != LAT),
         'corrected by fusion (word agent said Latin)': nonlat & (p['full'] != LAT) & (p['word'] == LAT),
         'non-Latin, never sent to Latin': nonlat & (p['full'] != LAT) & (p['word'] != LAT)}
    T_ = pd.DataFrame([(k, int(m.sum()), 100 * float(np.nanmean(sh_l[m])), 100 * float(np.nanmean(sh_b[m]))) for k, m in G.items()], columns=['group (' + VAR + ')', 'words', '% neighbours in line called Latin', '% neighbours in block called Latin'])
    display(T_.round(1)); T_.to_csv(f'{ROOT}/results/latin_pull_{VAR}.csv', index=False)
    p_after = p['gated']; print(VAR, 'non-Latin predicted Latin: word only %.2f %% | full fusion %.2f %% | full fusion with gate %.2f %%' % tuple(100 * ((q == LAT) & nonlat).sum() / nonlat.sum() for q in (p['word'], p['full'], p_after)))
print('If the newly wrong words have many more Latin-called neighbours than the others, the pull explanation is supported.')